In [1]:
import sys, os
# Erzwingt den Hauptpfad
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.db_connect import load_sql, get_engine

In [2]:
import sys
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sqlalchemy import text

# Pfad-Setup: Damit Python den 'src'-Ordner findet
sys.path.append(os.path.abspath('..'))
from src.db_connect import get_engine

# Engine erstellen: Die Verbindung zur Datenbank aufbauen
engine = get_engine()

In [3]:
query_peep_fio2 = text("""
SELECT 
    ce1.hadm_id, 
    ce1.charttime,
    ce1.valuenum as fio2,
    ce2.valuenum as peep
FROM chartevents ce1
JOIN chartevents ce2 ON ce1.hadm_id = ce2.hadm_id AND ce1.charttime = ce2.charttime
WHERE ce1.itemid IN (223835, 3420) -- FiO2 (Prozent oder Dezimal)
AND ce2.itemid IN (220339, 507)   -- PEEP
AND ce1.valuenum IS NOT NULL 
AND ce2.valuenum IS NOT NULL
""")

with engine.connect() as conn:
    df_vent = pd.read_sql(query_peep_fio2, conn)

In [4]:
def get_ards_soll_peep(fio2):
    # Umrechnung falls FiO2 in Prozent vorliegt (z.B. 50 statt 0.5)
    if fio2 > 1.0:
        fio2 = fio2 / 100.0
        
    # ARDSnet Low PEEP Tabelle
    if fio2 <= 0.3: return 5
    if fio2 <= 0.4: return 8
    if fio2 <= 0.5: return 10
    if fio2 <= 0.6: return 10
    if fio2 <= 0.7: return 12
    if fio2 <= 0.8: return 14
    if fio2 <= 0.9: return 16
    return 18 # bei FiO2 1.0

# Anwendung auf den Datensatz
df_vent['peep_soll'] = df_vent['fio2'].apply(get_ards_soll_peep)

# Berechnung der Abweichung (Delta)
# Positiver Wert = Mehr PEEP als empfohlen (Aggressiv)
# Negativer Wert = Weniger PEEP als empfohlen (Konservativ)
df_vent['peep_delta'] = df_vent['peep'] - df_vent['peep_soll']

In [5]:
print("--- Statistik der PEEP-Abweichung ---")
print(df_vent['peep_delta'].describe())

# Wie viele Messpunkte liegen außerhalb der Norm?
unterversorgt = (df_vent['peep_delta'] <= -3).sum()
ueberversorgt = (df_vent['peep_delta'] >= 3).sum()
total = len(df_vent)

print(f"\nUnter-PEEP (Delta <= -3): {unterversorgt} ({unterversorgt/total*100:.1f}%)")
print(f"Über-PEEP  (Delta >= 3):  {ueberversorgt} ({ueberversorgt/total*100:.1f}%)")

--- Statistik der PEEP-Abweichung ---
count    608504.000000
mean          1.389181
std           7.651217
min         -18.000000
25%          -3.000000
50%          -3.000000
75%           8.000000
max          75.000000
Name: peep_delta, dtype: float64

Unter-PEEP (Delta <= -3): 310947 (51.1%)
Über-PEEP  (Delta >= 3):  170368 (28.0%)


In [6]:
import pandas as pd
import numpy as np
from sqlalchemy import text

# 1. Daten frisch laden (PEEP, FiO2 und Mortalität)
query_vent_mort = text("""
SELECT 
    ce1.hadm_id, 
    ce1.valuenum as fio2,
    ce2.valuenum as peep,
    adm.hospital_expire_flag
FROM chartevents ce1
JOIN chartevents ce2 ON ce1.hadm_id = ce2.hadm_id AND ce1.charttime = ce2.charttime
JOIN admissions adm ON ce1.hadm_id = adm.hadm_id
WHERE ce1.itemid IN (223835, 3420) -- FiO2
AND ce2.itemid IN (220339, 507)   -- PEEP
AND ce1.valuenum IS NOT NULL 
AND ce2.valuenum IS NOT NULL
""")

with engine.connect() as conn:
    df_peep_study = pd.read_sql(query_vent_mort, conn)

# 2. PEEP-Soll Funktion (ARDSnet Low PEEP Tabelle)
def get_ards_soll(fio2):
    if fio2 > 1.0: fio2 /= 100.0 # Falls 50 statt 0.5
    if fio2 <= 0.3: return 5
    if fio2 <= 0.4: return 8
    if fio2 <= 0.5: return 10
    if fio2 <= 0.7: return 12
    if fio2 <= 0.8: return 14
    if fio2 <= 0.9: return 16
    return 18

# 3. Berechnungen durchführen
df_peep_study['peep_soll'] = df_peep_study['fio2'].apply(get_ards_soll)
df_peep_study['delta'] = df_peep_study['peep'] - df_peep_study['peep_soll']

# 4. Kategorisierung auf Patientenebene
# Wir berechnen erst den Durchschnitts-Delta pro Patient
df_patient = df_peep_study.groupby('hadm_id').agg({
    'delta': 'mean',
    'hospital_expire_flag': 'first'
}).reset_index()

def categorize(d):
    if d <= -3: return '1. Unter-PEEP (Gefahr: Atelektase)'
    if d >= 3:  return '3. Über-PEEP (Gefahr: Barotrauma)'
    return '2. Leitliniengerecht'

df_patient['gruppe'] = df_patient['delta'].apply(categorize)

# 5. Finale Ergebnistabelle
summary = df_patient.groupby('gruppe')['hospital_expire_flag'].agg(['mean', 'count'])
summary['mean'] *= 100
summary.columns = ['Sterberate %', 'Anzahl Patienten']

print("--- ERGEBNIS DER PEEP-ADHERENZ-ANALYSE ---")
display(summary.round(2))

--- ERGEBNIS DER PEEP-ADHERENZ-ANALYSE ---


,Sterberate %,Anzahl Patienten
gruppe,,
1. Unter-PEEP (Gefahr: Atelektase),15.46,7963
2. Leitliniengerecht,23.48,2070
3. Über-PEEP (Gefahr: Barotrauma),5.96,1275


In [7]:
from scipy.stats import chi2_contingency

# 1. Kreuztabelle (Contingency Table) erstellen
# Zeilen: Beatmungsgruppe, Spalten: Verstorben (0 = Nein, 1 = Ja)
contingency_table = pd.crosstab(df_patient['gruppe'], df_patient['hospital_expire_flag'])

# 2. Chi-Quadrat-Test durchführen
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

# 3. Ergebnisse schön aufbereiten
print("--- STATISTISCHE SIGNIFIKANZ: PEEP-STRATEGIE ---")
print(f"Chi-Quadrat-Statistik: {chi2:.4f}")
print(f"p-Wert:                {p_value:.4e}") # Wissenschaftliche Notation für sehr kleine Werte

if p_value < 0.05:
    print("\nERGEBNIS: Statistisch signifikant (p < 0.05)")
    print("Die Beatmungsstrategie hat einen messbaren Einfluss auf die Mortalität.")
else:
    print("\nERGEBNIS: Nicht signifikant (p >= 0.05)")
    print("Es konnte kein statistisch belegbarer Zusammenhang gefunden werden.")

# Optional: Die Kreuztabelle anzeigen, um die Verteilung zu sehen
print("\n--- Beobachtete Häufigkeiten ---")
display(contingency_table)

--- STATISTISCHE SIGNIFIKANZ: PEEP-STRATEGIE ---
Chi-Quadrat-Statistik: 184.6527
p-Wert:                8.0015e-41

ERGEBNIS: Statistisch signifikant (p < 0.05)
Die Beatmungsstrategie hat einen messbaren Einfluss auf die Mortalität.

--- Beobachtete Häufigkeiten ---


hospital_expire_flag,0,1
gruppe,,
1. Unter-PEEP (Gefahr: Atelektase),6732,1231
2. Leitliniengerecht,1584,486
3. Über-PEEP (Gefahr: Barotrauma),1199,76


In [11]:
print("Spalten in df_patient:", df_patient.columns)
print("Spalten in df_sofa:", df_sofa.columns)

Spalten in df_patient: Index(['hadm_id', 'delta', 'hospital_expire_flag', 'gruppe'], dtype='object')
Spalten in df_sofa: Index(['icustay_id', 'sofa_score', 'cardiovascular', 'respiration', 'renal'], dtype='object')


In [12]:
# 1. SQL-Abfrage anpassen: Wir holen die hadm_id dazu
query_sofa = text("""
SELECT 
    ie.hadm_id, 
    s.icustay_id, 
    s.sofa_score, 
    s.cardiovascular, 
    s.respiration, 
    s.renal 
FROM sofa s
JOIN icustays ie ON s.icustay_id = ie.icustay_id
""")

with engine.connect() as conn:
    df_sofa = pd.read_sql(query_sofa, conn)

# 2. Jetzt klappt der Merge über 'hadm_id'
df_analysis = pd.merge(df_patient, df_sofa, on='hadm_id', how='left')

# 3. Analyse durchführen
sofa_summary = df_analysis.groupby('gruppe').agg({
    'sofa_score': 'mean',
    'cardiovascular': 'mean',
    'hospital_expire_flag': 'mean'
}).rename(columns={'hospital_expire_flag': 'Sterberate'})

sofa_summary['Sterberate'] = (sofa_summary['Sterberate'] * 100).round(2)

print("--- ANALYSE: KRANKHEITSSCHWERE PRO PEEP-GRUPPE ---")
display(sofa_summary)

--- ANALYSE: KRANKHEITSSCHWERE PRO PEEP-GRUPPE ---


,sofa_score,cardiovascular,Sterberate
gruppe,,,
1. Unter-PEEP (Gefahr: Atelektase),5.270895,1.388977,16.08
2. Leitliniengerecht,6.039323,1.684144,24.61
3. Über-PEEP (Gefahr: Barotrauma),5.527387,0.108764,5.94
